In [ ]:
from IPython.core.display import display, HTML
display(HTML('<style>.container {with:98% !important;}</style>'))

import os
import glob
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import pickle

import itertools
import math
from scipy import signal
from scipy.stats import norm
import statsmodels.api as sm
from sqlalchemy import create_engine
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix
from sklearn.utils import shuffle

warnings.filterwarnings('ignore')


np.random.seed(566)


In [ ]:
def find_first_position_label(numbers,param,label_list,range1,min_length,org_labels, set_seg):
    consecutive_cont = 0
    start_index = None
    end_index = None
    first_pos = []
    arr = np.array(numbers[param])
#     arr[arr==-32767] = np.nan
#     print(len(arr),len(numbers[param]))
    
    for i,num in enumerate(arr):
#         print(num[param])
        if (num < range1) and (num>0):
            if consecutive_cont == 0:
                start_index = i
            consecutive_cont+=1
        else:
            if consecutive_cont >=  min_length:
                end_index = i-1
                first_pos.append(start_index)
                first_pos.append(end_index)
#                 print(start_index,end_index)
            consecutive_cont = 0
            start_index = None
            end_index = None
            
    filter_pos = []
    for i in range(0,len(first_pos)-1,2):
        is_add = 1
        for p_num in range(0,i,2):
            prev_num = first_pos[p_num]
            if abs(first_pos[i]-prev_num) <= set_seg:
                is_add= 0
                break
        if is_add == 1:
            filter_pos.append(first_pos[i])
            filter_pos.append(first_pos[i+1])
    
    for idx in range(0,len(filter_pos)-1,2):
        group = filter_pos[idx]
        group2 = filter_pos[idx+1]
        data_id = numbers['data_id'][group]
        label_time = numbers['label_time'][group]
        event_time_from = label_time
        event_time_to = numbers['label_time'][group2]
        rank = 'X'
        log = param+str(range1)+'_last1min'
        label_list.append([data_id, event_time_from, event_time_to, label_time, rank, log])

In [ ]:
data_list :['data_id']
data_path : vital sign path
data_info : ['record_start_time','record_end_time','label_time','data_id']

no_vital_file = []
data_cout_time = []
set_seg = [300]
for set_len in set_seg:
    print(set_len)
    label_info = []
    for idx,rd in tqdm(data_list.iterrows()):
        data_id = rd['data_id']
        # print(data_id)
        if data_info[data_info['data_id'] == str(data_id)].empty:
            continue
        record_start_time = data_info[data_info['data_id'] == data_id]['record_start_time'].iloc[0]
        record_start_time = pd.to_datetime(record_start_time)
        if os.path.isfile(data_path+'/'+data_id+'_vital.csv'):
            file = pd.read_csv(data_path+'/'+data_id+'_vital.csv')
        else:
            no_vital_file.append(data_id)
            print('not_exist_vitalsign',data_id)
            continue
        data_cout_time.append([data_id,file.shape[0]/60/60])
        file['label_time'] = pd.date_range(start=record_start_time + pd.to_timedelta('{}s'.format(1)),
                                       periods=file.shape[0],
                                       freq='{}s'.format(1))
        file['data_id'] = data_id
        org_labels = []
        temp_data = file[['data_id','MAP','SPO2','label_time']]
        column_name = ['data_id', 'event_time_from', 'event_time_to', 'label_time', 'rank', 'log']
        label_list = []
        find_first_position_label(temp_data,'MAP',label_list,65, 60,org_labels,set_len)
        label_list = pd.DataFrame(label_list,columns=column_name)
        label_info.append(label_list)
        label_list = []
        find_first_position_label(temp_data,'SPO2',label_list,90, 60,org_labels,set_len)
        label_list = pd.DataFrame(label_list,columns=column_name)
        label_info.append(label_list)
    if len(label_info)>0:
        label_info = pd.concat(label_info)
    label_info = pd.DataFrame(label_info)
    label_info = label_info.drop_duplicates()
    label_info.to_excel(r'lowsbpspo2_label.xlsx',index =False)